In [1]:
import sys
import gc
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

import models
import replay_buffer
import train
import evaluation

importlib.reload(models)
importlib.reload(replay_buffer)
importlib.reload(train)
importlib.reload(evaluation)

from models import DQN
from train import ConfigDQN, entrenar_dqn
from evaluation import evaluar_modelo

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo: {device}")

Dispositivo: mps


# Ajuste de hiperparámetros de DQN + PER + n-step

En este notebook se analiza el efecto de dos hiperparámetros:

- El grado de priorización del replay buffer, controlado por `per_alpha`.
- El número de pasos utilizados para acumular recompensas, controlado por `n_step`.

Los demás hiperparámetros se mantendrán constantes para realizar una comparación controlada.

In [2]:
import gc

# Liberar modelos grandes que ya no necesitamos en memoria
objetos_temporales = [
    "resultado_dqn_per_3step",
    "mejor_dqn_per_3step",
    "checkpoint_3step",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_per_alpha04_3step = ConfigDQN(
    nombre_experimento="v7_dqn_per_a04_3step",
    total_pasos=1_000_000,

    seed=42,
    gamma=0.99,
    n_step=3,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.4,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_per_alpha04_3step = entrenar_dqn(
    config=config_per_alpha04_3step,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print(
    "\nENTRENAMIENTO V7 FINALIZADO"
)

print(
    "Mejor promedio de evaluación: "
    f"{resultado_per_alpha04_3step['mejor_promedio_evaluacion']:.2f}"
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Replay buffer: priorizado | alpha=0.4 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 10,000/1,000,000 | episodio=49 | epsilon=0.964 | loss=0.0550 | Q=0.006
Paso 11,000/1,000,000 | episodio=53 | epsilon=0.960 | loss=0.0331 | Q=0.128
Paso 12,000/1,000,000 | episodio=59 | epsilon=0.957 | loss=0.0270 | Q=0.138
Paso 13,000/1,000,000 | episodio=62 | epsilon=0.953 | loss=0.0308 | Q=0.126
Paso 14,000/1,000,000 | episodio=67 | epsilon=0.950 | loss=0.0261 | Q=0.158
Paso 15,000/1,000,000 | episodio=71 | epsilon=0.946 | loss=0.0160 | Q=0.170
Paso 16,000/1,000,000 | episodio=77 | epsilon=0.942 | loss=0.0417 | Q=0.188
Paso 17,000/1,000,000 | episodio=81 | epsilon=0.939 | loss=0.0172 | Q=0.149
Paso 18,000/1,000,000 | episodio=84 | epsilon=0.935 | loss=0.0445 | Q=0.160
Paso 19,000/1,000,000 | episodio=88 | epsilon=0.932 | loss=0.0268 | Q=0.192
Paso 20,000/1,000,000 | episodio=92 | epsilon=0.928 | loss=0.0418 | Q=0.218
Paso 21,000/1,000,000 | episodio=97 | epsilon=0.924 | loss=0.0049 | Q=0.253
Paso 

In [3]:
ruta_evaluaciones_v7 = Path(
    "../logs/entrenamientos/"
    "v7_dqn_per_a04_3step/evaluaciones.csv"
)

df_evaluaciones_v7 = pd.read_csv(
    ruta_evaluaciones_v7
)

df_evaluaciones_v7 = (
    df_evaluaciones_v7
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_v7)

mejor_evaluacion_v7 = df_evaluaciones_v7.loc[
    df_evaluaciones_v7["promedio"].idxmax()
]

ultimas_tres_v7 = (
    df_evaluaciones_v7
    .tail(3)["promedio"]
    .mean()
)

print("\nRESUMEN DE V7")
print(
    f"Mejor paso: "
    f"{int(mejor_evaluacion_v7['paso_global']):,}"
)
print(
    f"Mejor promedio: "
    f"{mejor_evaluacion_v7['promedio']:.2f}"
)
print(
    f"Mediana del mejor checkpoint: "
    f"{mejor_evaluacion_v7['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_v7['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_v7['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_v7['maximo']:.2f}"
)
print(
    f"Promedio de las últimas tres evaluaciones: "
    f"{ultimas_tres_v7:.2f}"
)

ruta_mejor_v7 = Path(
    "../models/v7_dqn_per_a04_3step/"
    "mejor_modelo.pt"
)

print(
    "\nCheckpoint disponible: "
    f"{'OK' if ruta_mejor_v7.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,50000,149.0,135.0,52.478567,95.0,240.0
1,100000,191.0,160.0,116.206712,70.0,330.0
2,150000,297.0,295.0,116.730459,135.0,460.0
3,200000,383.0,305.0,123.393679,255.0,545.0
4,250000,302.0,305.0,80.659779,175.0,420.0
5,300000,242.0,160.0,113.604577,155.0,445.0
6,350000,250.0,200.0,85.029407,160.0,365.0
7,400000,478.0,465.0,124.923977,325.0,630.0
8,450000,267.0,220.0,113.428392,130.0,425.0
9,500000,294.0,330.0,103.121288,160.0,415.0



RESUMEN DE V7
Mejor paso: 1,000,000
Mejor promedio: 493.00
Mediana del mejor checkpoint: 480.00
Desviación: 114.04
Mínimo: 365.00
Máximo: 705.00
Promedio de las últimas tres evaluaciones: 438.67

Checkpoint disponible: OK
